In [ ]:
!pip install torch torchvision torchaudio
!pip install torch-geometric

In [ ]:
import os, glob

print("=== PATH DIAGNOSTIC ===")
for p in [
    '/kaggle/input/notebooks/zaharch/quantum-machine-9-qm9/data.covs.pickle',
    '/kaggle/input/champs-scalar-coupling/structures.csv',
]:
    print(f"\n{p}")
    print(f"  exists: {os.path.isfile(p)}")

print("\n=== ALL FILES UNDER /kaggle/input ===")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import json, pandas as pd

# Print the notebook source to see how the pickle was built
nb = json.load(open('/kaggle/input/notebooks/zaharch/quantum-machine-9-qm9/__notebook__.ipynb'))
for cell in nb['cells']:
    if cell['cell_type'] == 'code':
        src = ''.join(cell['source'])
        # Only print cells that mention positions/structures/xyz/pickle
        if any(k in src for k in ['pickle', 'struct', 'position', 'xyz', 'x_1', 'pos']):
            print('='*60)
            print(src[:2000])

# Also print ALL columns in the pickle
df = pd.read_pickle('/kaggle/input/notebooks/zaharch/quantum-machine-9-qm9/data.covs.pickle')
print('\nALL COLUMNS:')
print(df.columns.tolist())
print('\nSHAPE:', df.shape)
print('\nFIRST ROW:')
print(df.iloc[0].to_dict())

In [ ]:
import json
nb = json.load(open('/kaggle/input/notebooks/zaharch/quantum-machine-9-qm9/__notebook__.ipynb'))
# Print datasources the notebook was run with
print(nb.get('metadata', {}).get('kernelspec', ''))
print(nb.get('metadata', {}).get('dataSources', ''))
# Also check if any xyz content is embedded in outputs
for i, cell in enumerate(nb['cells']):
    for output in cell.get('outputs', []):
        text = ''.join(output.get('text', []))
        if 'xyz' in text.lower() or 'structure' in text.lower() or 'dsgdb' in text.lower():
            print(f'Cell {i}:', text[:500])

In [ ]:
import os, glob

# Check if zaharch QM9 aka dataset is mounted separately
for path in glob.glob('/kaggle/input/**', recursive=True):
    if 'dsgdb' in path or 'qm9' in path.lower():
        print(path)

In [ ]:
# =============================================================================
# E(n) Equivariant Graph Neural Networks (EGNN) — QM9 Molecular Property Prediction
# Paper : https://proceedings.mlr.press/v139/satorras21a/satorras21a.pdf
# Repo  : https://github.com/vgsatorras/egnn
#
# ── WHAT YOU HAVE ────────────────────────────────────────────────────────────
#
#   /kaggle/input/notebooks/zaharch/quantum-machine-9-qm9/data.covs.pickle
#       DataFrame (7 M rows). Columns: molecule_name, homo, lumo, gap, mu,
#       alpha, r2, zpve, U0, U, H, G, Cv, mulliken_*, freqs_*, ...
#       NO x/y/z coordinates — geometry was never stored in this pickle.
#
# ── WHAT YOU NEED TO ADD ─────────────────────────────────────────────────────
#
#   The zaharch "Quantum Machine 9, aka QM9" dataset which contains the
#   133,885 individual xyz files with atom positions.
#
#   In your Kaggle notebook:
#     1. Click "Add data" (top right)
#     2. Search: zaharch quantum machine 9
#     3. Add "Quantum Machine 9, aka QM9" by zaharch
#     4. Re-run the notebook
#
#   Files will appear at:
#     /kaggle/input/quantum-machine-9-aka-qm9/dsgdb9nsd.xyz/dsgdb9nsd_000001.xyz
#
# ── LOADING STRATEGY ─────────────────────────────────────────────────────────
#
#   1. Pickle  → groupby molecule_name → one row per molecule with QM9 props
#   2. xyz dir → parse positions + atom types for each molecule
#   3. Join    → complete molecule dict → train EGNN
#
# =============================================================================

import os, json, time, random, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# =============================================================================
# SECTION 1 — EGNN MODEL
# =============================================================================

def unsorted_segment_sum(data, segment_ids, num_segments):
    result = data.new_full((num_segments, data.size(1)), 0)
    idx = segment_ids.unsqueeze(-1).expand(-1, data.size(1))
    result.scatter_add_(0, idx, data)
    return result

def unsorted_segment_mean(data, segment_ids, num_segments):
    result = data.new_full((num_segments, data.size(1)), 0)
    count  = data.new_full((num_segments, data.size(1)), 0)
    idx    = segment_ids.unsqueeze(-1).expand(-1, data.size(1))
    result.scatter_add_(0, idx, data)
    count.scatter_add_(0, idx, torch.ones_like(data))
    return result / count.clamp(min=1)


class E_GCL(nn.Module):
    def __init__(self, input_nf, output_nf, hidden_nf,
                 edges_in_d=0, act_fn=nn.SiLU(),
                 residual=True, attention=False,
                 normalize=False, coords_agg='mean', tanh=False):
        super().__init__()
        self.residual   = residual
        self.attention  = attention
        self.normalize  = normalize
        self.coords_agg = coords_agg
        self.epsilon    = 1e-8

        self.edge_mlp = nn.Sequential(
            nn.Linear(input_nf * 2 + 1 + edges_in_d, hidden_nf), act_fn,
            nn.Linear(hidden_nf, hidden_nf), act_fn,
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_nf + input_nf, hidden_nf), act_fn,
            nn.Linear(hidden_nf, output_nf),
        )
        coord_layers = [nn.Linear(hidden_nf, hidden_nf), act_fn]
        last = nn.Linear(hidden_nf, 1, bias=False)
        nn.init.xavier_uniform_(last.weight, gain=0.001)
        coord_layers.append(last)
        if tanh:
            coord_layers.append(nn.Tanh())
        self.coord_mlp = nn.Sequential(*coord_layers)
        if attention:
            self.att_mlp = nn.Sequential(nn.Linear(hidden_nf, 1), nn.Sigmoid())

    def coord2radial(self, edge_index, coord):
        row, col = edge_index
        diff   = coord[row] - coord[col]
        radial = torch.sum(diff ** 2, dim=1, keepdim=True)
        if self.normalize:
            diff = diff / (torch.sqrt(radial).detach() + self.epsilon)
        return radial, diff

    def edge_model(self, source, target, radial, edge_attr):
        parts = [source, target, radial] + ([edge_attr] if edge_attr is not None else [])
        out = self.edge_mlp(torch.cat(parts, dim=1))
        if self.attention:
            out = out * self.att_mlp(out)
        return out

    def coord_model(self, coord, edge_index, coord_diff, edge_feat):
        row, _ = edge_index
        trans  = coord_diff * self.coord_mlp(edge_feat)
        agg_fn = unsorted_segment_sum if self.coords_agg == 'sum' else unsorted_segment_mean
        return coord + agg_fn(trans, row, coord.size(0))

    def node_model(self, x, edge_index, edge_attr, node_attr):
        row, _ = edge_index
        agg  = unsorted_segment_sum(edge_attr, row, x.size(0))
        parts = [x, agg] + ([node_attr] if node_attr is not None else [])
        out  = self.node_mlp(torch.cat(parts, dim=1))
        return (x + out) if self.residual else out

    def forward(self, h, edge_index, coord, edge_attr=None, node_attr=None):
        row, col           = edge_index
        radial, coord_diff = self.coord2radial(edge_index, coord)
        edge_feat          = self.edge_model(h[row], h[col], radial, edge_attr)
        coord              = self.coord_model(coord, edge_index, coord_diff, edge_feat)
        h                  = self.node_model(h, edge_index, edge_feat, node_attr)
        return h, coord, edge_attr


class EGNN_QM9(nn.Module):
    def __init__(self, in_node_nf, in_edge_nf, hidden_nf,
                 device='cpu', act_fn=nn.SiLU(), n_layers=7, attention=True):
        super().__init__()
        self.n_layers  = n_layers
        self.embedding = nn.Linear(in_node_nf, hidden_nf)
        for i in range(n_layers):
            self.add_module(f"gcl_{i}", E_GCL(
                hidden_nf, hidden_nf, hidden_nf,
                edges_in_d=in_edge_nf, act_fn=act_fn,
                residual=True, attention=attention,
            ))
        self.node_dec = nn.Sequential(
            nn.Linear(hidden_nf, hidden_nf), act_fn,
            nn.Linear(hidden_nf, hidden_nf),
        )
        self.graph_dec = nn.Sequential(
            nn.Linear(hidden_nf, hidden_nf), act_fn,
            nn.Linear(hidden_nf, 1),
        )
        self.to(device)

    def forward(self, h0, x, edges, edge_attr=None, node_mask=None, n_nodes=None):
        h = self.embedding(h0)
        for i in range(self.n_layers):
            h, x, _ = self._modules[f"gcl_{i}"](h, edges, x, edge_attr=edge_attr)
        h = self.node_dec(h)
        if node_mask is not None:
            h = h * node_mask
        B = h.size(0) // n_nodes
        h = h.view(B, n_nodes, -1)
        if node_mask is not None:
            mask = node_mask.view(B, n_nodes, 1)
            h = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        else:
            h = h.mean(1)
        return self.graph_dec(h).squeeze(-1)


_adj_cache = {}

def get_adj_matrix(n_nodes, batch_size, device):
    key = (n_nodes, batch_size, str(device))
    if key not in _adj_cache:
        rows, cols = zip(*[(i, j) for i in range(n_nodes)
                                   for j in range(n_nodes) if i != j])
        r = torch.LongTensor(rows)
        c = torch.LongTensor(cols)
        all_r = torch.cat([r + n_nodes * i for i in range(batch_size)])
        all_c = torch.cat([c + n_nodes * i for i in range(batch_size)])
        _adj_cache[key] = [all_r.to(device), all_c.to(device)]
    return _adj_cache[key]


# =============================================================================
# SECTION 2 — GEOMETRY: locate the zaharch QM9 xyz files
#
# The pickle has no coordinates. Geometry comes from the zaharch dataset:
#   "Quantum Machine 9, aka QM9" by zaharch on Kaggle
#
# HOW TO ADD IT (do this once, then re-run):
#   1. Open your Kaggle notebook
#   2. Click "Add data" (top right)
#   3. Search: zaharch quantum machine 9
#   4. Add "Quantum Machine 9, aka QM9" by zaharch
#   → Files will appear at /kaggle/input/quantum-machine-9-aka-qm9/dsgdb9nsd.xyz/
#
# =============================================================================

ATOM_ENCODER  = {'H': 0, 'C': 1, 'N': 2, 'O': 3, 'F': 4}
NUM_ATOM_TYPES = len(ATOM_ENCODER)
SYM_TO_Z      = {'H': 1, 'C': 6, 'N': 7, 'O': 8, 'F': 9}
QM9_PROPS     = ['mu','alpha','homo','lumo','gap','r2','zpve','U0','U','H','G','Cv']

# Known Kaggle mount path for zaharch/quantum-machine-9-aka-qm9
_KNOWN_XYZ_DIR = '/kaggle/input/quantum-machine-9-aka-qm9/dsgdb9nsd.xyz'


def _find_xyz_dir():
    """
    Return the directory containing dsgdb9nsd_*.xyz files.
    Checks the known Kaggle path first, then does a quick scan of /kaggle/input.
    Raises a clear error with instructions if not found.
    """
    # 1. Exact known path after adding zaharch dataset via Kaggle UI
    if (os.path.isdir(_KNOWN_XYZ_DIR) and
            len(glob.glob(os.path.join(_KNOWN_XYZ_DIR, 'dsgdb9nsd_*.xyz'))) > 100):
        return _KNOWN_XYZ_DIR

    # 2. Scan all of /kaggle/input for any folder with dsgdb9nsd_*.xyz files
    for dirpath, _, filenames in os.walk('/kaggle/input'):
        n = sum(1 for f in filenames
                if f.startswith('dsgdb9nsd_') and f.endswith('.xyz'))
        if n > 100:
            return dirpath

    # 3. Not found — raise with clear instructions
    raise FileNotFoundError(
        "\n\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        "  QM9 xyz files not found. The pickle has no geometry.\n"
        "  You must add the zaharch dataset to this notebook:\n\n"
        "  1. Click 'Add data' in the top-right of your Kaggle notebook\n"
        "  2. Search: zaharch quantum machine 9\n"
        "  3. Add 'Quantum Machine 9, aka QM9' by zaharch\n"
        "  4. Re-run the notebook\n\n"
        "  After adding, files will be at:\n"
        f"  {_KNOWN_XYZ_DIR}/dsgdb9nsd_000001.xyz\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    )


def _parse_float(s):
    return float(s.replace('*^', 'e'))


def _parse_xyz_positions(path):
    """
    Parse ONLY atom symbols and positions from a QM9 xyz file.
    Returns (symbols: list[str], positions: ndarray(N,3)) or (None, None).
    """
    try:
        with open(path) as f:
            lines = f.readlines()
        na   = int(lines[0].strip())
        syms, pos = [], []
        for i in range(na):
            toks = lines[2 + i].strip().split()
            syms.append(toks[0])
            pos.append([_parse_float(toks[1]),
                        _parse_float(toks[2]),
                        _parse_float(toks[3])])
        return syms, np.array(pos, dtype=np.float32)
    except Exception:
        return None, None


# =============================================================================
# SECTION 3 — LOAD DATASET
#
# Strategy:
#   1. Read the pickle → group by molecule_name → one row per molecule
#      with all QM9 scalar properties.
#   2. For each molecule, read its xyz file (downloaded above) for geometry.
#   3. Build a list of molecule dicts: positions, charges, properties.
# =============================================================================

PICKLE_PATH = '/kaggle/input/notebooks/zaharch/quantum-machine-9-qm9/data.covs.pickle'


def load_dataset(pickle_path=PICKLE_PATH, max_samples=None, verbose=True):
    """
    Load QM9 molecules by joining:
      - QM9 scalar properties  from the zaharch pickle
      - Atom positions (x,y,z) from the zaharch xyz files

    Returns a list of dicts with keys:
        n_atoms, positions (N,3) float32, charges (N,) float32,
        homo, lumo, gap, mu, alpha, r2, zpve, U0, U, H, G, Cv
    """
    # ── Step 1: locate xyz directory (raises if not found) ────────────────────
    xyz_dir = _find_xyz_dir()
    n_xyz = len(glob.glob(os.path.join(xyz_dir, 'dsgdb9nsd_*.xyz')))
    if verbose:
        print(f"  QM9 xyz files: {xyz_dir}  ({n_xyz:,} files)")

    # ── Step 2: load QM9 properties from pickle ───────────────────────────────
    if not os.path.isfile(pickle_path):
        raise FileNotFoundError(f"Pickle not found: {pickle_path}")

    if verbose:
        print(f"  Reading pickle: {pickle_path}")
    df = pd.read_pickle(pickle_path)

    prop_cols = [c for c in QM9_PROPS if c in df.columns]
    # One row per molecule — properties are identical for every row of a molecule
    mol_props = df.groupby('molecule_name')[prop_cols].first().reset_index()
    if verbose:
        print(f"  {len(df):,} coupling rows → {len(mol_props):,} unique molecules")
        print(f"  Properties available: {prop_cols}")

    if max_samples:
        mol_props = mol_props.iloc[:max_samples]

    # ── Step 3: parse xyz for each molecule → join positions + properties ─────
    if verbose:
        print(f"  Parsing xyz geometry …")

    data_list, n_skipped = [], 0

    for _, row in mol_props.iterrows():
        name  = row['molecule_name']           # e.g. "dsgdb9nsd_000001"
        fpath = os.path.join(xyz_dir, name + '.xyz')

        if not os.path.isfile(fpath):
            n_skipped += 1
            continue

        syms, positions = _parse_xyz_positions(fpath)
        if syms is None:
            n_skipped += 1
            continue

        # Skip molecules with atoms outside H/C/N/O/F
        if not set(syms).issubset(ATOM_ENCODER):
            n_skipped += 1
            continue

        charges = np.array([float(SYM_TO_Z[s]) for s in syms], dtype=np.float32)

        mol = {
            'n_atoms'  : len(syms),
            'positions': positions,
            'charges'  : charges,
        }
        for p in QM9_PROPS:
            mol[p] = float(row[p]) if p in prop_cols else 0.0

        data_list.append(mol)

        if verbose and len(data_list) % 10000 == 0:
            print(f"    … {len(data_list):,} molecules loaded")

    if verbose:
        print(f"  Done. {len(data_list):,} molecules "
              f"(skipped {n_skipped} missing/unsupported)\n")
    return data_list


def _synthetic_fallback(n=500):
    data_list = []
    for _ in range(n):
        na = np.random.randint(3, 10)
        mol = {'n_atoms': na,
               'positions': np.random.randn(na, 3).astype(np.float32),
               'charges'  : np.random.choice([1,6,7,8], size=na).astype(np.float32)}
        for p in QM9_PROPS:
            mol[p] = float(np.random.randn())
        data_list.append(mol)
    return data_list


# =============================================================================
# SECTION 4 — PYTORCH DATASET
# =============================================================================

class QM9Dataset(Dataset):
    def __init__(self, data_list, property='homo', max_atoms=29):
        self.data     = data_list
        self.property = property
        self.N        = max_atoms

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        mol = self.data[idx]
        n, N = mol['n_atoms'], self.N

        pos      = np.zeros((N, 3), dtype=np.float32)
        one_hot  = np.zeros((N, NUM_ATOM_TYPES), dtype=np.float32)
        charges  = np.zeros((N, 1), dtype=np.float32)
        atom_mask= np.zeros((N, 1), dtype=np.float32)

        pos[:n]        = mol['positions'][:n]
        charges[:n, 0] = mol['charges'][:n]
        atom_mask[:n, 0] = 1.0

        z_to_sym = {1:'H', 6:'C', 7:'N', 8:'O', 9:'F'}
        for i, z in enumerate(mol['charges'][:n]):
            sym = z_to_sym.get(int(z), 'C')
            if sym in ATOM_ENCODER:
                one_hot[i, ATOM_ENCODER[sym]] = 1.0

        return {
            'positions' : torch.FloatTensor(pos),
            'one_hot'   : torch.FloatTensor(one_hot),
            'charges'   : torch.FloatTensor(charges),
            'atom_mask' : torch.FloatTensor(atom_mask),
            self.property: torch.FloatTensor([mol[self.property]]),
        }


def split_dataset(data_list, train_size=110000, val_size=10000,
                  test_size=10000, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(data_list))
    return ([data_list[i] for i in idx[:train_size]],
            [data_list[i] for i in idx[train_size:train_size+val_size]],
            [data_list[i] for i in idx[train_size+val_size:train_size+val_size+test_size]])


# =============================================================================
# SECTION 5 — NODE FEATURE PREPROCESSING
# =============================================================================

def preprocess_input(one_hot, charges, charge_power, charge_scale, device):
    if one_hot.dim() == 3:
        one_hot = one_hot.view(-1, one_hot.size(-1))
    if charges.dim() == 3:
        charges = charges.view(-1, 1)
    charge_tensor = (charges / charge_scale).pow(
        torch.arange(1, charge_power + 1, device=device, dtype=torch.float32))
    return torch.cat([one_hot, charge_tensor], dim=1)


def compute_mean_mad(loader, prop_name):
    values = torch.cat([b[prop_name] for b in loader]).float()
    mean   = values.mean().item()
    mad    = (values - mean).abs().mean().item()
    print(f"  Property '{prop_name}':  mean={mean:.6f}  MAD={mad:.6f}")
    return mean, mad


# =============================================================================
# SECTION 6 — TRAINING & EVALUATION
# =============================================================================

def _forward(model, batch, n_nodes, charge_power, charge_scale, device):
    B   = batch['positions'].size(0)
    pos = batch['positions'].view(B * n_nodes, -1).to(device)
    mask= batch['atom_mask'].view(B * n_nodes, -1).to(device)
    nodes = preprocess_input(
        batch['one_hot'].to(device),
        batch['charges'].to(device),
        charge_power, charge_scale, device
    ).view(B * n_nodes, -1)
    edges = get_adj_matrix(n_nodes, B, device)
    return model(h0=nodes, x=pos, edges=edges, node_mask=mask, n_nodes=n_nodes)


def train_epoch(model, loader, optimizer, loss_fn, mean, mad,
                prop, charge_power, charge_scale, n_nodes, device):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        optimizer.zero_grad()
        pred  = _forward(model, batch, n_nodes, charge_power, charge_scale, device)
        label = batch[prop].to(device).squeeze(-1)
        loss  = loss_fn(pred, (label - mean) / mad)
        loss.backward(); optimizer.step()
        total += loss.item() * batch['positions'].size(0)
        n     += batch['positions'].size(0)
    return total / n


@torch.no_grad()
def eval_epoch(model, loader, loss_fn, mean, mad,
               prop, charge_power, charge_scale, n_nodes, device):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        pred  = _forward(model, batch, n_nodes, charge_power, charge_scale, device)
        label = batch[prop].to(device).squeeze(-1)
        total += loss_fn(mad * pred + mean, label).item() * batch['positions'].size(0)
        n     += batch['positions'].size(0)
    return total / n


# =============================================================================
# SECTION 7 — TEST EVALUATION WITH PERCENTAGE ERROR
# =============================================================================

@torch.no_grad()
def eval_test_detailed(model, loader, mean, mad,
                       prop, charge_power, charge_scale, n_nodes, device):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        pred  = _forward(model, batch, n_nodes, charge_power, charge_scale, device)
        label = batch[prop].to(device).squeeze(-1)
        preds.append((mad * pred + mean).cpu())
        labels.append(label.cpu())

    preds  = torch.cat(preds)
    labels = torch.cat(labels)
    abs_err = (preds - labels).abs()
    rel_err = abs_err / labels.abs().clamp(min=1e-8) * 100.0
    n = len(labels)
    return dict(
        mae          = abs_err.mean().item(),
        rmse         = (preds - labels).pow(2).mean().sqrt().item(),
        mape         = rel_err.mean().item(),
        within_1pct  = (rel_err <  1.0).sum().item() / n * 100,
        within_5pct  = (rel_err <  5.0).sum().item() / n * 100,
        within_10pct = (rel_err < 10.0).sum().item() / n * 100,
        n_total      = n,
    )


# =============================================================================
# SECTION 8 — MAIN
# =============================================================================

def main():
    PROPERTY     = 'homo'   # mu alpha homo lumo gap r2 zpve U0 U H G Cv
    LR           = 1e-3
    BATCH_SIZE   = 96
    EPOCHS       = 1000
    N_LAYERS     = 7
    HIDDEN_NF    = 128
    ATTENTION    = True
    CHARGE_POWER = 2        # in_node_nf = 5 + 2 = 7
    MAX_ATOMS    = 29
    NUM_WORKERS  = 2
    WEIGHT_DECAY = 1e-16

    # ── Load data ─────────────────────────────────────────────────────────────
    if os.path.isfile(PICKLE_PATH):
        all_data = load_dataset(verbose=True)
    else:
        print(f"⚠  Pickle not found at:\n   {PICKLE_PATH}\n"
              "   Running on synthetic data.\n")
        all_data = _synthetic_fallback(500)

    n_total = len(all_data)
    if n_total >= 130000:
        _tr, _va, _te = 110000, 10000, 10000
    else:
        _tr = max(BATCH_SIZE, int(n_total * 0.8))
        _va = max(BATCH_SIZE, int(n_total * 0.1))
        _te = max(BATCH_SIZE, n_total - _tr - _va)

    train_data, val_data, test_data = split_dataset(
        all_data, train_size=_tr, val_size=_va, test_size=_te)
    print(f"Split → train={len(train_data):,}  val={len(val_data):,}  test={len(test_data):,}\n")

    mk = lambda d, s: DataLoader(
        QM9Dataset(d, property=PROPERTY, max_atoms=MAX_ATOMS),
        batch_size=BATCH_SIZE, shuffle=s, num_workers=NUM_WORKERS,
        drop_last=(s and len(d) > BATCH_SIZE))
    loaders = dict(train=mk(train_data, True),
                   valid=mk(val_data,   False),
                   test =mk(test_data,  False))

    charge_scale = max(float(max(d['charges'].max() for d in train_data)), 9.0)

    print("Dataset statistics:")
    mean, mad = compute_mean_mad(loaders['train'], PROPERTY)
    print()

    in_node_nf = NUM_ATOM_TYPES + CHARGE_POWER   # 7
    model = EGNN_QM9(in_node_nf=in_node_nf, in_edge_nf=0, hidden_nf=HIDDEN_NF,
                     device=DEVICE, act_fn=nn.SiLU(),
                     n_layers=N_LAYERS, attention=ATTENTION)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: EGNN_QM9  ({n_params:,} trainable parameters)")
    print(f"Target: {PROPERTY}  |  Epochs: {EPOCHS}  |  LR: {LR}\n")

    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
    loss_l1   = nn.L1Loss()

    os.makedirs('/kaggle/working/egnn_logs', exist_ok=True)
    best_val, best_epoch, history = float('inf'), 0, []

    print(f"{'Epoch':>6}  {'Train MAE':>10}  {'Val MAE':>10}  {'Best Val':>10}  {'Time':>6}")
    print("─" * 55)

    for epoch in range(1, EPOCHS + 1):
        t0        = time.time()
        train_mae = train_epoch(model, loaders['train'], optimizer, loss_l1,
                                mean, mad, PROPERTY, CHARGE_POWER,
                                charge_scale, MAX_ATOMS, DEVICE)
        scheduler.step()
        val_mae   = eval_epoch(model, loaders['valid'], loss_l1,
                               mean, mad, PROPERTY, CHARGE_POWER,
                               charge_scale, MAX_ATOMS, DEVICE)

        history.append({'epoch': epoch, 'train_mae': train_mae, 'val_mae': val_mae})

        if val_mae < best_val:
            best_val, best_epoch = val_mae, epoch
            torch.save(model.state_dict(), '/kaggle/working/egnn_best.pt')

        print(f"{epoch:>6}  {train_mae:>10.5f}  {val_mae:>10.5f}  "
              f"{best_val:>10.5f}  {time.time()-t0:>5.1f}s")

    with open('/kaggle/working/egnn_logs/history.json', 'w') as f:
        json.dump(history, f, indent=2)

    # ── Test evaluation ───────────────────────────────────────────────────────
    print(f"\n{'═'*55}")
    print(f"Training done. Loading best checkpoint (epoch {best_epoch}) …")
    model.load_state_dict(torch.load('/kaggle/working/egnn_best.pt', map_location=DEVICE))

    res = eval_test_detailed(model, loaders['test'], mean, mad,
                             PROPERTY, CHARGE_POWER, charge_scale, MAX_ATOMS, DEVICE)
    n = res['n_total']
    print(f"\n{'═'*55}")
    print(f"  TEST RESULTS  —  '{PROPERTY}'  ({n:,} molecules)")
    print(f"{'═'*55}")
    print(f"  MAE                          : {res['mae']:.6f}")
    print(f"  RMSE                         : {res['rmse']:.6f}")
    print(f"  MAPE (mean abs % error)      : {res['mape']:.2f} %")
    print(f"{'─'*55}")
    print(f"  Molecules within  1% error   : {res['within_1pct']:.1f} %")
    print(f"  Molecules within  5% error   : {res['within_5pct']:.1f} %")
    print(f"  Molecules within 10% error   : {res['within_10pct']:.1f} %")
    print(f"{'═'*55}")
    print(f"  Best validation MAE          : {best_val:.6f}  (epoch {best_epoch})")
    print(f"{'═'*55}\n")

    summary = dict(**res, best_val_mae=best_val,
                   best_val_epoch=best_epoch, property=PROPERTY)
    with open('/kaggle/working/egnn_logs/test_results.json', 'w') as f:
        json.dump(summary, f, indent=2)


# =============================================================================
# SECTION 9 — SMOKE TEST
# =============================================================================

def smoke_test():
    print("── Smoke test: EGNN_QM9 ────────────────────────────────────────────")
    model = EGNN_QM9(in_node_nf=7, in_edge_nf=0, hidden_nf=64,
                     device='cpu', n_layers=2, attention=True)
    N, B = 10, 2
    pred = model(h0=torch.randn(B*N, 7), x=torch.randn(B*N, 3),
                 edges=get_adj_matrix(N, B, 'cpu'),
                 node_mask=torch.ones(B*N, 1), n_nodes=N)
    assert pred.shape == (B,)
    print(f"  pred shape: {pred.shape}  ✓")
    print("  All checks passed ✓\n")


if __name__ == "__main__":
    smoke_test()
    main()
else:
    smoke_test()

In [ ]:
# =============================================================================
# E(3) Equivariant Diffusion Model (EDM) — QM9 Molecule Generation
# Paper : https://proceedings.mlr.press/v162/hoogeboom22a/hoogeboom22a.pdf
# Based on: https://github.com/ehoogeboom/e3_diffusion_for_molecules
#
# Self-contained, no rdkit, Python 3.12 compatible, Kaggle-ready.
# Same data loading as EGNN reference code (zaharch QM9 dataset).
#
# Dataset paths (add zaharch dataset via "Add data" in Kaggle):
#   xyz : /kaggle/input/quantum-machine-9-aka-qm9/dsgdb9nsd.xyz/
#   pkl : /kaggle/input/notebooks/zaharch/quantum-machine-9-qm9/data.covs.pickle
#
# Key design decisions vs the original repo:
#   - No rdkit dependency anywhere (rdkit_functions.py not used)
#   - Time embedding: sinusoidal (32-dim) appended to node features at every step
#   - Fully-connected per-molecule edges built via a cached adj-matrix (same trick
#     as EGNN reference code, adapted for padded batches with edge masking)
#   - EMA on model weights (decay = 0.9999) evaluated on validation set
#   - Polynomial-2 noise schedule (paper default)
#   - Output: per-epoch train/val diffusion loss (L2), same table format as EGNN
# =============================================================================

import os, glob, time, json, math, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# =============================================================================
# SECTION 1 — BUILDING BLOCKS
# =============================================================================

def unsorted_segment_sum(data, segment_ids, num_segments):
    result = data.new_zeros(num_segments, data.size(1))
    idx    = segment_ids.unsqueeze(-1).expand_as(data)
    result.scatter_add_(0, idx, data)
    return result


def unsorted_segment_mean(data, segment_ids, num_segments):
    result = data.new_zeros(num_segments, data.size(1))
    count  = data.new_zeros(num_segments, data.size(1))
    idx    = segment_ids.unsqueeze(-1).expand_as(data)
    result.scatter_add_(0, idx, data)
    count.scatter_add_(0, idx, torch.ones_like(data))
    return result / count.clamp(min=1)


class GCL(nn.Module):
    """Graph Convolutional Layer: edge MLP + node MLP."""

    def __init__(self, input_nf, output_nf, hidden_nf,
                 edges_in_d=2, act_fn=nn.SiLU(), attention=True,
                 aggregation_method='sum', normalization_factor=1.0):
        super().__init__()
        self.aggregation_method   = aggregation_method
        self.normalization_factor = normalization_factor
        self.attention            = attention

        self.edge_mlp = nn.Sequential(
            nn.Linear(input_nf * 2 + edges_in_d, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, hidden_nf),
            act_fn,
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_nf + input_nf, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, output_nf),
        )
        if attention:
            self.att_mlp = nn.Sequential(nn.Linear(hidden_nf, 1), nn.Sigmoid())

    def edge_model(self, h_i, h_j, edge_attr, edge_mask):
        out = self.edge_mlp(torch.cat([h_i, h_j, edge_attr], dim=1))
        if self.attention:
            out = out * self.att_mlp(out)
        if edge_mask is not None:
            out = out * edge_mask
        return out

    def node_model(self, h, edge_index, m_ij):
        row = edge_index[0]
        if self.aggregation_method == 'sum':
            agg = unsorted_segment_sum(m_ij, row, h.size(0))
        else:
            agg = unsorted_segment_mean(m_ij, row, h.size(0))
        agg = agg / self.normalization_factor
        return self.node_mlp(torch.cat([h, agg], dim=1))

    def forward(self, h, edge_index, edge_attr, edge_mask=None):
        row, col = edge_index
        m_ij     = self.edge_model(h[row], h[col], edge_attr, edge_mask)
        h_out    = self.node_model(h, edge_index, m_ij)
        return h_out, m_ij


class EquivariantCoordUpdate(nn.Module):
    """Coordinate update: x_i ← x_i + Σ_j (x_i − x_j) · φ_x(m_ij)."""

    def __init__(self, hidden_nf, act_fn=nn.SiLU(), tanh=True,
                 coords_range=15.0, aggregation_method='sum',
                 normalization_factor=1.0):
        super().__init__()
        self.tanh                 = tanh
        self.coords_range         = coords_range
        self.aggregation_method   = aggregation_method
        self.normalization_factor = normalization_factor

        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_nf, hidden_nf), act_fn,
            nn.Linear(hidden_nf, 1, bias=False),
        )
        if tanh:
            # append tanh after the last linear
            self.coord_mlp = nn.Sequential(
                nn.Linear(hidden_nf, hidden_nf), act_fn,
                nn.Linear(hidden_nf, 1, bias=False),
                nn.Tanh(),
            )
        nn.init.xavier_uniform_(self.coord_mlp[-2 if tanh else -1].weight, gain=0.001)

    def forward(self, x, edge_index, coord_diff, m_ij, edge_mask=None):
        row = edge_index[0]
        weights = self.coord_mlp(m_ij)       # (E, 1)
        if self.tanh:
            weights = weights * self.coords_range
        trans = coord_diff * weights         # (E, 3)
        if edge_mask is not None:
            trans = trans * edge_mask
        if self.aggregation_method == 'sum':
            agg = unsorted_segment_sum(trans, row, x.size(0))
        else:
            agg = unsorted_segment_mean(trans, row, x.size(0))
        return x + agg / self.normalization_factor


class EGNN_Layer(nn.Module):
    """One full E(n)-equivariant layer: messages + coord update + node update."""

    def __init__(self, hidden_nf, edges_in_d=2, act_fn=nn.SiLU(),
                 attention=True, norm_diff=True, tanh=True,
                 coords_range=15.0, aggregation_method='sum',
                 normalization_factor=1.0):
        super().__init__()
        self.norm_diff = norm_diff

        self.gcl = GCL(
            hidden_nf, hidden_nf, hidden_nf,
            edges_in_d=edges_in_d, act_fn=act_fn,
            attention=attention, aggregation_method=aggregation_method,
            normalization_factor=normalization_factor,
        )
        self.coord_update = EquivariantCoordUpdate(
            hidden_nf, act_fn=act_fn, tanh=tanh,
            coords_range=coords_range,
            aggregation_method=aggregation_method,
            normalization_factor=normalization_factor,
        )

    def forward(self, h, x, edge_index, node_mask=None, edge_mask=None):
        row, col   = edge_index
        coord_diff = x[row] - x[col]                      # (E, 3)
        radial     = (coord_diff ** 2).sum(1, keepdim=True)  # (E, 1)

        if self.norm_diff:
            norm       = (radial.sqrt() + 1e-8).detach()
            coord_diff = coord_diff / norm

        # Edge features: normalised radial + raw radial/(raw+1)
        edge_attr = torch.cat([radial / (radial + 1.0), radial], dim=1)  # (E, 2)
        if edge_mask is not None:
            edge_attr = edge_attr * edge_mask

        h, m_ij = self.gcl(h, edge_index, edge_attr, edge_mask)
        x       = self.coord_update(x, edge_index, coord_diff, m_ij, edge_mask)

        if node_mask is not None:
            h = h * node_mask
        return h, x


# =============================================================================
# SECTION 2 — EGNN DYNAMICS (denoising network)
#
# Takes (z_t, t) where z_t = [x_t | h_t] and outputs predicted noise ε̂.
# Time is a sinusoidal embedding appended to node features.
# =============================================================================

class EGNN_Dynamics(nn.Module):
    """
    EGNN denoising backbone for EDM.

    in_node_nf  : dimensionality of h (node features, NOT including time emb).
    time_emb_dim: sinusoidal time embedding size; appended to h before first layer.
    """

    def __init__(self, in_node_nf, time_emb_dim=32, hidden_nf=256,
                 n_layers=9, act_fn=nn.SiLU(), attention=True,
                 norm_diff=True, tanh=True, coords_range=15.0,
                 aggregation_method='sum', normalization_factor=1.0,
                 n_dims=3):
        super().__init__()
        self.n_dims      = n_dims
        self.n_layers    = n_layers
        self.in_node_nf  = in_node_nf
        self.time_emb_dim= time_emb_dim

        # Input embedding: h + time_emb → hidden
        self.node_embedding = nn.Linear(in_node_nf + time_emb_dim, hidden_nf)
        # Output head: hidden → h (predict noise for features only; coord noise
        # is implicit via the equivariant coord updates inside the layers)
        self.node_output    = nn.Linear(hidden_nf, in_node_nf)

        for i in range(n_layers):
            self.add_module(f"layer_{i}", EGNN_Layer(
                hidden_nf, edges_in_d=2, act_fn=act_fn,
                attention=attention, norm_diff=norm_diff, tanh=tanh,
                coords_range=coords_range,
                aggregation_method=aggregation_method,
                normalization_factor=normalization_factor,
            ))

    @staticmethod
    def _sinusoidal_time_emb(t_int, T, dim, device):
        """
        t_int : (B,) integer steps
        Returns (B, dim) sinusoidal embeddings.
        """
        half = dim // 2
        freq = math.log(10000) / (half - 1)
        freq = torch.exp(torch.arange(half, device=device).float() * -freq)
        t_f  = t_int.float() / T                  # (B,)
        emb  = t_f[:, None] * freq[None, :]       # (B, half)
        return torch.cat([emb.sin(), emb.cos()], dim=1)  # (B, dim)

    def forward(self, h, x, t_int, T, edge_index, node_mask=None, edge_mask=None):
        """
        h         : (B*N, in_node_nf)
        x         : (B*N, 3)
        t_int     : (B,) integer time steps
        T         : total diffusion steps (scalar)
        edge_index: [row, col]
        node_mask : (B*N, 1) or None
        edge_mask : (E, 1) or None

        Returns:
          h_eps : (B*N, in_node_nf) — predicted feature noise
          x_eps : (B*N, 3)          — predicted coordinate noise (= x_start - x / sigma after caller)
                                      Actually: updated coordinates (caller computes eps from these)
        """
        B_N = h.size(0)
        B   = t_int.size(0)
        N   = B_N // B

        # Time embedding (B, time_emb_dim) → broadcast to (B*N, time_emb_dim)
        t_emb = self._sinusoidal_time_emb(t_int, T, self.time_emb_dim, h.device)
        t_emb = t_emb.repeat_interleave(N, dim=0)   # (B*N, time_emb_dim)

        # Embed node features + time
        h_in = torch.cat([h, t_emb], dim=1)         # (B*N, in_node_nf + time_emb_dim)
        h    = self.node_embedding(h_in)             # (B*N, hidden_nf)
        if node_mask is not None:
            h = h * node_mask

        x_in = x.clone()   # save input coords for residual

        for i in range(self.n_layers):
            h, x = self._modules[f"layer_{i}"](
                h, x, edge_index, node_mask=node_mask, edge_mask=edge_mask)

        # CoM-free coordinate output: predicted noise for coords
        # We return Δx = x - x_in (the equivariant update is the coord noise)
        x_eps = x - x_in

        # Remove CoM drift from predicted coord noise (per-molecule)
        x_eps = _remove_mean_with_mask_batched(x_eps, node_mask, B, N)

        # Feature output
        h_eps = self.node_output(h)   # (B*N, in_node_nf)
        if node_mask is not None:
            h_eps = h_eps * node_mask

        return h_eps, x_eps


def _remove_mean_with_mask_batched(x, node_mask, B, N):
    """
    Remove per-molecule CoM from x.
    x         : (B*N, 3)
    node_mask : (B*N, 1)
    """
    x_r = x.view(B, N, 3)
    m_r = node_mask.view(B, N, 1) if node_mask is not None else torch.ones_like(x_r[:, :, :1])
    com = (x_r * m_r).sum(1, keepdim=True) / m_r.sum(1, keepdim=True).clamp(min=1)
    x_r = x_r - com
    if node_mask is not None:
        x_r = x_r * m_r
    return x_r.view(B * N, 3)


# =============================================================================
# SECTION 3 — EDGE INDEX CACHE
# (Fully-connected within each molecule, with edge masking for padding)
# =============================================================================

_adj_cache = {}

def get_adj_matrix(n_nodes, batch_size, device):
    """Pre-built fully-connected (no self-loops) edge index, cached."""
    key = (n_nodes, batch_size, str(device))
    if key not in _adj_cache:
        rows, cols = zip(*[(i, j) for i in range(n_nodes)
                                   for j in range(n_nodes) if i != j])
        r = torch.LongTensor(rows)
        c = torch.LongTensor(cols)
        all_r = torch.cat([r + n_nodes * k for k in range(batch_size)])
        all_c = torch.cat([c + n_nodes * k for k in range(batch_size)])
        _adj_cache[key] = [all_r.to(device), all_c.to(device)]
    return _adj_cache[key]


def make_edge_mask(node_mask, edge_index):
    """edge_mask[e] = 1 iff both endpoint atoms are real (not padding)."""
    row, col   = edge_index
    edge_mask  = node_mask[row] * node_mask[col]   # (E, 1)
    return edge_mask


# =============================================================================
# SECTION 4 — NOISE SCHEDULES
# =============================================================================

def polynomial_schedule(timesteps, s=1e-5, power=2.0):
    """
    Polynomial-2 schedule (EDM paper default).
    Returns alpha_bar tensor of shape (T+1,) with alpha_bar[0]=1, alpha_bar[T]~0.
    """
    T     = timesteps
    steps = T + 1
    t     = torch.linspace(0, T, steps)
    # Signal variance: (1 - (t/T)^p)^2, clipped and renormalised
    alpha2 = (1.0 - (t / T) ** power) ** 2
    alpha2 = alpha2.clamp(min=s)
    alpha2 = alpha2 / alpha2[0]          # ensure alpha2[0] = 1
    return alpha2.sqrt()                 # return alpha_bar (not alpha_bar²)


def cosine_schedule(timesteps, s=0.008):
    T     = timesteps
    steps = T + 1
    t     = torch.linspace(0, T, steps)
    alpha_bar = torch.cos(((t / T) + s) / (1.0 + s) * math.pi * 0.5) ** 2
    alpha_bar = alpha_bar / alpha_bar[0]
    return alpha_bar.sqrt().clamp(min=0.0, max=1.0)


# =============================================================================
# SECTION 5 — EDM CORE
#
# Forward process: q(z_t | z_0) = N(alpha_bar_t * z_0,  sigma_bar_t^2 * I)
#   alpha_bar_t² + sigma_bar_t² = 1  (variance-preserving)
#
# Training objective (simple L2 on noise):
#   L = E_{t, z_0, eps} [ || eps - eps_theta(z_t, t) ||² ]
#
# Denoising prediction: network predicts eps directly (eps-parameterisation).
# =============================================================================

class EDM(nn.Module):
    """E(3) Equivariant Diffusion Model for QM9 molecule generation."""

    def __init__(self,
                 in_node_nf     = 6,          # 5 atom types + 1 charge feat
                 hidden_nf      = 256,
                 n_layers        = 9,
                 timesteps       = 1000,
                 noise_schedule  = 'polynomial_2',
                 noise_precision = 1e-5,
                 time_emb_dim    = 32,
                 attention       = True,
                 norm_values     = (1., 4., 10.),
                 n_dims          = 3,
                 device          = 'cpu'):
        super().__init__()

        self.in_node_nf    = in_node_nf
        self.n_dims        = n_dims
        self.T             = timesteps
        self.norm_values   = norm_values
        self.time_emb_dim  = time_emb_dim
        self.device_str    = str(device)

        # Build denoising network
        self.dynamics = EGNN_Dynamics(
            in_node_nf           = in_node_nf,
            time_emb_dim         = time_emb_dim,
            hidden_nf            = hidden_nf,
            n_layers             = n_layers,
            act_fn               = nn.SiLU(),
            attention            = attention,
            norm_diff            = True,
            tanh                 = True,
            coords_range         = 15.0,
            aggregation_method   = 'sum',
            normalization_factor = 1.0,
            n_dims               = n_dims,
        )

        # Noise schedule buffers: shape (T+1,)
        if noise_schedule == 'cosine':
            alpha_bar = cosine_schedule(timesteps, s=noise_precision)
        elif noise_schedule.startswith('polynomial'):
            power     = float(noise_schedule.split('_')[1]) if '_' in noise_schedule else 2.0
            alpha_bar = polynomial_schedule(timesteps, s=noise_precision, power=power)
        else:
            raise ValueError(f"Unknown noise schedule: {noise_schedule}")

        sigma_bar = (1.0 - alpha_bar ** 2).clamp(min=0.0).sqrt()

        self.register_buffer('alpha_bar', alpha_bar)  # (T+1,)
        self.register_buffer('sigma_bar', sigma_bar)  # (T+1,)

        # Posterior variance for DDPM sampling:
        # beta_tilde_t = sigma²_{t-1} / sigma²_t * (1 - alpha²_t / alpha²_{t-1})
        alpha2 = alpha_bar ** 2
        sigma2 = sigma_bar ** 2
        beta_t = torch.zeros(timesteps + 1)
        for t in range(1, timesteps + 1):
            num = sigma2[t - 1]
            den = sigma2[t].clamp(min=1e-10)
            fac = (1.0 - alpha2[t] / alpha2[t - 1].clamp(min=1e-10)).clamp(min=0.0)
            beta_t[t] = (num / den * fac).clamp(max=0.999)
        self.register_buffer('beta_t', beta_t)

        self.to(device)

    # ──────────────────────────────────────────────────────────────────────────
    # Normalisation (applied once before adding noise, undone after)
    # ──────────────────────────────────────────────────────────────────────────

    def _norm_x(self, x):   return x / self.norm_values[0]
    def _norm_h(self, h):   return h / self.norm_values[1]
    def _unnorm_x(self, x): return x * self.norm_values[0]
    def _unnorm_h(self, h): return h * self.norm_values[1]

    # ──────────────────────────────────────────────────────────────────────────
    # Training loss
    # ──────────────────────────────────────────────────────────────────────────

    def compute_loss(self, x0, h0, t_int, node_mask, edge_index, edge_mask, B, N):
        """
        x0        : (B*N, 3)         clean normalised coordinates
        h0        : (B*N, in_node_nf) clean normalised node features
        t_int     : (B,) LongTensor  time step index ∈ [1, T]
        node_mask : (B*N, 1)
        edge_index: [row, col] prebuilt fully-connected adj
        edge_mask : (E, 1)
        """
        # Broadcast alpha/sigma to atom level
        a   = self.alpha_bar[t_int]                      # (B,)
        s   = self.sigma_bar[t_int]                      # (B,)
        a_n = a.repeat_interleave(N).unsqueeze(1)        # (B*N, 1)
        s_n = s.repeat_interleave(N).unsqueeze(1)        # (B*N, 1)

        # Sample noise
        eps_x = torch.randn_like(x0)
        eps_h = torch.randn_like(h0)

        # Zero-CoM on coordinate noise (EDM paper Section 3.1)
        eps_x = _remove_mean_with_mask_batched(eps_x, node_mask, B, N)

        # Forward noising
        z_x = a_n * x0 + s_n * eps_x
        z_h = a_n * h0 + s_n * eps_h
        z_x = z_x * node_mask
        z_h = z_h * node_mask

        # Predict noise via denoising network
        # dynamics returns (h_eps, x_eps) — features first, coords second
        eps_h_pred, eps_x_pred = self.dynamics(
            h=z_h, x=z_x, t_int=t_int, T=self.T,
            edge_index=edge_index, node_mask=node_mask, edge_mask=edge_mask,
        )

        # L2 loss on noise prediction, masked
        loss_x = ((eps_x - eps_x_pred) ** 2) * node_mask
        loss_h = ((eps_h - eps_h_pred) ** 2) * node_mask

        # Weight by number of spatial + feature dims
        n_active = node_mask.sum().clamp(min=1)
        loss = (loss_x.sum() * self.n_dims + loss_h.sum() * self.in_node_nf) / \
               (n_active * (self.n_dims + self.in_node_nf))
        return loss

    # ──────────────────────────────────────────────────────────────────────────
    # DDPM reverse sampling
    # ──────────────────────────────────────────────────────────────────────────

    @torch.no_grad()
    def sample(self, n_samples, n_nodes, device):
        """Generate n_samples molecules each with n_nodes atoms (all unmasked)."""
        B = n_samples
        N = n_nodes
        node_mask  = torch.ones(B * N, 1, device=device)
        edge_index = get_adj_matrix(N, B, device)
        edge_mask  = make_edge_mask(node_mask, edge_index)

        # Start from z_T ~ N(0, I)
        z_x = torch.randn(B * N, self.n_dims, device=device)
        z_h = torch.randn(B * N, self.in_node_nf, device=device)
        z_x = _remove_mean_with_mask_batched(z_x, node_mask, B, N)

        for t_val in reversed(range(1, self.T + 1)):
            t_int = torch.full((B,), t_val, device=device, dtype=torch.long)
            a  = self.alpha_bar[t_val]
            s  = self.sigma_bar[t_val]
            a0 = self.alpha_bar[t_val - 1]
            s0 = self.sigma_bar[t_val - 1]

            # dynamics returns (h_eps, x_eps) — features first, coords second
            eps_h, eps_x = self.dynamics(
                h=z_h, x=z_x, t_int=t_int, T=self.T,
                edge_index=edge_index, node_mask=node_mask, edge_mask=edge_mask,
            )

            # Predict z_0
            z0_x = (z_x - s * eps_x) / a.clamp(min=1e-8)
            z0_h = (z_h - s * eps_h) / a.clamp(min=1e-8)

            # DDPM posterior mean
            mu_x = a0 * (s ** 2) / (s ** 2 + (a * s0) ** 2 + 1e-10) * z0_x + \
                   a  * (s0 ** 2) / (s ** 2 + (a * s0) ** 2 + 1e-10) * z_x
            mu_h = a0 * (s ** 2) / (s ** 2 + (a * s0) ** 2 + 1e-10) * z0_h + \
                   a  * (s0 ** 2) / (s ** 2 + (a * s0) ** 2 + 1e-10) * z_h

            noise_scale = self.beta_t[t_val].sqrt()
            if t_val > 1:
                z_x = mu_x + noise_scale * torch.randn_like(mu_x)
                z_h = mu_h + noise_scale * torch.randn_like(mu_h)
            else:
                z_x, z_h = mu_x, mu_h

            z_x = _remove_mean_with_mask_batched(z_x, node_mask, B, N)

        # Unnormalise
        z_x = self._unnorm_x(z_x)
        z_h = self._unnorm_h(z_h)
        return z_x.view(B, N, 3), z_h.view(B, N, self.in_node_nf)


# =============================================================================
# SECTION 6 — EMA
# =============================================================================

class EMA:
    """Exponential Moving Average of model weights."""

    def __init__(self, model, decay=0.9999):
        self.model  = model
        self.decay  = decay
        self.shadow = {k: v.clone().float()
                       for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self):
        d = self.decay
        for k, v in self.model.state_dict().items():
            if self.shadow[k].dtype.is_floating_point:
                self.shadow[k] = d * self.shadow[k] + (1.0 - d) * v.float()

    def apply_shadow(self):
        self._backup = copy.deepcopy(self.model.state_dict())
        new_sd = {k: v.to(self.model.state_dict()[k].device).to(self.model.state_dict()[k].dtype)
                  for k, v in self.shadow.items()}
        self.model.load_state_dict(new_sd)

    def restore(self):
        self.model.load_state_dict(self._backup)


# =============================================================================
# SECTION 7 — DATA LOADING  (same as EGNN reference)
# =============================================================================

ATOM_ENCODER   = {'H': 0, 'C': 1, 'N': 2, 'O': 3, 'F': 4}
NUM_ATOM_TYPES = len(ATOM_ENCODER)
SYM_TO_Z       = {'H': 1, 'C': 6, 'N': 7, 'O': 8, 'F': 9}
QM9_PROPS      = ['mu','alpha','homo','lumo','gap','r2','zpve','U0','U','H','G','Cv']

PICKLE_PATH    = '/kaggle/input/notebooks/zaharch/quantum-machine-9-qm9/data.covs.pickle'
_KNOWN_XYZ_DIR = '/kaggle/input/quantum-machine-9-aka-qm9/dsgdb9nsd.xyz'


def _find_xyz_dir():
    if (os.path.isdir(_KNOWN_XYZ_DIR) and
            len(glob.glob(os.path.join(_KNOWN_XYZ_DIR, 'dsgdb9nsd_*.xyz'))) > 100):
        return _KNOWN_XYZ_DIR
    for dirpath, _, filenames in os.walk('/kaggle/input'):
        n = sum(1 for f in filenames
                if f.startswith('dsgdb9nsd_') and f.endswith('.xyz'))
        if n > 100:
            return dirpath
    raise FileNotFoundError(
        
        f"  Files should appear at: {_KNOWN_XYZ_DIR}/dsgdb9nsd_000001.xyz\n"
    )


def _parse_float(s):
    return float(s.replace('*^', 'e'))


def _parse_xyz(path):
    try:
        with open(path) as f:
            lines = f.readlines()
        na   = int(lines[0].strip())
        syms, pos = [], []
        for i in range(na):
            toks = lines[2 + i].strip().split()
            syms.append(toks[0])
            pos.append([_parse_float(toks[1]),
                        _parse_float(toks[2]),
                        _parse_float(toks[3])])
        return syms, np.array(pos, dtype=np.float32)
    except Exception:
        return None, None


def load_dataset(pickle_path=PICKLE_PATH, max_samples=None, verbose=True):
    xyz_dir = _find_xyz_dir()
    n_xyz   = len(glob.glob(os.path.join(xyz_dir, 'dsgdb9nsd_*.xyz')))
    if verbose:
        print(f"  QM9 xyz files: {xyz_dir}  ({n_xyz:,} files)")

    if not os.path.isfile(pickle_path):
        raise FileNotFoundError(f"Pickle not found: {pickle_path}")

    if verbose:
        print(f"  Reading pickle: {pickle_path}")
    df = pd.read_pickle(pickle_path)

    prop_cols = [c for c in QM9_PROPS if c in df.columns]
    mol_props = df.groupby('molecule_name')[prop_cols].first().reset_index()
    if verbose:
        print(f"  {len(df):,} coupling rows → {len(mol_props):,} unique molecules")
        print(f"  Properties available: {prop_cols}")

    if max_samples:
        mol_props = mol_props.iloc[:max_samples]

    if verbose:
        print(f"  Parsing xyz geometry …")

    data_list, n_skipped = [], 0

    for _, row in mol_props.iterrows():
        name  = row['molecule_name']
        fpath = os.path.join(xyz_dir, name + '.xyz')

        if not os.path.isfile(fpath):
            n_skipped += 1
            continue

        syms, positions = _parse_xyz(fpath)
        if syms is None or not set(syms).issubset(ATOM_ENCODER):
            n_skipped += 1
            continue

        charges = np.array([float(SYM_TO_Z[s]) for s in syms], dtype=np.float32)
        mol = {'n_atoms': len(syms), 'positions': positions, 'charges': charges}
        for p in QM9_PROPS:
            mol[p] = float(row[p]) if p in prop_cols else 0.0
        data_list.append(mol)

        if verbose and len(data_list) % 10000 == 0:
            print(f"    … {len(data_list):,} molecules loaded")

    if verbose:
        print(f"  Done. {len(data_list):,} molecules "
              f"(skipped {n_skipped} missing/unsupported)\n")
    return data_list


def _synthetic_fallback(n=200):
    data_list = []
    for _ in range(n):
        na  = np.random.randint(3, 10)
        mol = {'n_atoms': na,
               'positions': np.random.randn(na, 3).astype(np.float32),
               'charges':   np.random.choice([1,6,7,8], size=na).astype(np.float32)}
        for p in QM9_PROPS:
            mol[p] = float(np.random.randn())
        data_list.append(mol)
    return data_list


def split_dataset(data_list, train_size=110000, val_size=10000,
                  test_size=10000, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(data_list))
    return (
        [data_list[i] for i in idx[:train_size]],
        [data_list[i] for i in idx[train_size:train_size + val_size]],
        [data_list[i] for i in idx[train_size + val_size:train_size + val_size + test_size]],
    )


# =============================================================================
# SECTION 8 — PYTORCH DATASET
# =============================================================================

class QM9DatasetEDM(Dataset):
    def __init__(self, data_list, max_atoms=29):
        self.data = data_list
        self.N    = max_atoms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        mol = self.data[idx]
        n, N = mol['n_atoms'], self.N

        pos       = np.zeros((N, 3), np.float32)
        one_hot   = np.zeros((N, NUM_ATOM_TYPES), np.float32)
        charges   = np.zeros((N, 1), np.float32)
        atom_mask = np.zeros((N, 1), np.float32)

        pos[:n]          = mol['positions'][:n]
        charges[:n, 0]   = mol['charges'][:n]
        atom_mask[:n, 0] = 1.0

        z_to_sym = {1:'H', 6:'C', 7:'N', 8:'O', 9:'F'}
        for i, z in enumerate(mol['charges'][:n]):
            sym = z_to_sym.get(int(z), 'C')
            if sym in ATOM_ENCODER:
                one_hot[i, ATOM_ENCODER[sym]] = 1.0

        return {
            'positions': torch.FloatTensor(pos),
            'one_hot'  : torch.FloatTensor(one_hot),
            'charges'  : torch.FloatTensor(charges),
            'atom_mask': torch.FloatTensor(atom_mask),
        }


# =============================================================================
# SECTION 9 — TRAINING & EVALUATION
# =============================================================================

def _forward_loss(model, batch, max_atoms, device):
    B   = batch['positions'].size(0)
    N   = max_atoms

    x         = batch['positions'].to(device).view(B * N, 3)
    one_hot   = batch['one_hot'].to(device).view(B * N, NUM_ATOM_TYPES)
    charges   = batch['charges'].to(device).view(B * N, 1)
    node_mask = batch['atom_mask'].to(device).view(B * N, 1)

    # Centre coordinates per molecule
    x = _remove_mean_with_mask_batched(x, node_mask, B, N)

    # Build node features: [one_hot | normalised_charge]
    h = torch.cat([one_hot, charges / 9.0], dim=1)   # (B*N, 6)

    # Normalise
    x_norm = x   / model.norm_values[0]
    h_norm = h   / model.norm_values[1]
    x_norm = x_norm * node_mask
    h_norm = h_norm * node_mask

    # Prebuilt fully-connected edges + edge mask
    edge_index = get_adj_matrix(N, B, device)
    edge_mask  = make_edge_mask(node_mask, edge_index)

    # Random time steps
    t_int = torch.randint(1, model.T + 1, (B,), device=device)

    return model.compute_loss(
        x0=x_norm, h0=h_norm, t_int=t_int,
        node_mask=node_mask,
        edge_index=edge_index, edge_mask=edge_mask,
        B=B, N=N,
    )


def train_epoch(model, loader, optimizer, ema, max_atoms, device):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        optimizer.zero_grad()
        loss = _forward_loss(model, batch, max_atoms, device)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if ema is not None:
            ema.update()
        total += loss.item() * batch['positions'].size(0)
        n     += batch['positions'].size(0)
    return total / n


@torch.no_grad()
def eval_epoch(model, loader, max_atoms, device, ema=None):
    if ema is not None:
        ema.apply_shadow()
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        loss   = _forward_loss(model, batch, max_atoms, device)
        total += loss.item() * batch['positions'].size(0)
        n     += batch['positions'].size(0)
    if ema is not None:
        ema.restore()
    return total / n


# =============================================================================
# SECTION 10 — SMOKE TEST
# =============================================================================

def smoke_test():
    print("── Smoke test: EDM ─────────────────────────────────────────────────")
    model = EDM(
        in_node_nf=6, hidden_nf=64, n_layers=2,
        timesteps=10, noise_schedule='polynomial_2',
        attention=True, device='cpu',
    )

    B, N = 2, 10
    x         = torch.randn(B * N, 3)
    one_hot   = F.one_hot(torch.randint(0, 5, (B * N,)), 5).float()
    charges   = torch.randint(1, 9, (B * N, 1)).float()
    node_mask = torch.ones(B * N, 1)
    h         = torch.cat([one_hot, charges / 9.0], dim=1)

    edge_index = get_adj_matrix(N, B, 'cpu')
    edge_mask  = make_edge_mask(node_mask, edge_index)
    t_int      = torch.randint(1, 11, (B,))

    loss = model.compute_loss(
        x0=x / model.norm_values[0],
        h0=h / model.norm_values[1],
        t_int=t_int,
        node_mask=node_mask,
        edge_index=edge_index, edge_mask=edge_mask,
        B=B, N=N,
    )
    assert loss.shape == torch.Size([]), f"Expected scalar, got {loss.shape}"
    assert not torch.isnan(loss), "Loss is NaN!"
    print(f"  loss: {loss.item():.5f}  ✓")
    print("  All checks passed ✓\n")


# =============================================================================
# SECTION 11 — MAIN
# =============================================================================

def main():
    # ── Hyperparameters (paper defaults; tuned for Kaggle GPU memory) ─────────
    DIFFUSION_STEPS  = 1000
    NOISE_SCHEDULE   = 'polynomial_2'
    NOISE_PRECISION  = 1e-5
    HIDDEN_NF        = 256
    N_LAYERS         = 9
    ATTENTION        = True
    LR               = 1e-4
    BATCH_SIZE       = 128       # paper: 64; fits ~16 GB GPU with N=29, nf=256
    EPOCHS           = 3000
    EMA_DECAY        = 0.9999
    MAX_ATOMS        = 29
    NUM_WORKERS      = 2
    NORM_VALUES      = (1., 4., 10.)
    IN_NODE_NF       = NUM_ATOM_TYPES + 1   # 5 one-hot + 1 normalised charge = 6
    TIME_EMB_DIM     = 32

    # ── Load data ─────────────────────────────────────────────────────────────
    if os.path.isfile(PICKLE_PATH):
        all_data = load_dataset(verbose=True)
    else:
        print(f"⚠  Pickle not found at:\n   {PICKLE_PATH}\n"
              "   Running on synthetic data.\n")
        all_data = _synthetic_fallback(200)

    n_total = len(all_data)
    if n_total >= 130000:
        _tr, _va, _te = 110000, 10000, 10000
    else:
        _tr = max(BATCH_SIZE, int(n_total * 0.8))
        _va = max(BATCH_SIZE, int(n_total * 0.1))
        _te = max(BATCH_SIZE, n_total - _tr - _va)

    train_data, val_data, test_data = split_dataset(
        all_data, train_size=_tr, val_size=_va, test_size=_te)
    print(f"Split → train={len(train_data):,}  val={len(val_data):,}  test={len(test_data):,}\n")

    mk = lambda d, s: DataLoader(
        QM9DatasetEDM(d, max_atoms=MAX_ATOMS),
        batch_size=BATCH_SIZE, shuffle=s,
        num_workers=NUM_WORKERS, pin_memory=True,
        drop_last=(s and len(d) > BATCH_SIZE),
    )
    loaders = dict(
        train = mk(train_data, True),
        valid = mk(val_data,   False),
        test  = mk(test_data,  False),
    )

    # ── Build model ───────────────────────────────────────────────────────────
    model = EDM(
        in_node_nf     = IN_NODE_NF,
        hidden_nf      = HIDDEN_NF,
        n_layers       = N_LAYERS,
        timesteps      = DIFFUSION_STEPS,
        noise_schedule = NOISE_SCHEDULE,
        noise_precision= NOISE_PRECISION,
        time_emb_dim   = TIME_EMB_DIM,
        attention      = ATTENTION,
        norm_values    = NORM_VALUES,
        n_dims         = 3,
        device         = DEVICE,
    )
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: EDM  ({n_params:,} trainable parameters)")
    print(f"Diffusion steps: {DIFFUSION_STEPS}  |  Schedule: {NOISE_SCHEDULE}  |"
          f"  Epochs: {EPOCHS}  |  LR: {LR}\n")

    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
    ema       = EMA(model, decay=EMA_DECAY)

    os.makedirs('/kaggle/working/edm_logs', exist_ok=True)
    best_val, best_epoch, history = float('inf'), 0, []

    print(f"{'Epoch':>6}  {'Train Loss':>11}  {'Val Loss':>11}  {'Best Val':>11}  {'Time':>6}")
    print("─" * 58)

    for epoch in range(1, EPOCHS + 1):
        t0         = time.time()
        train_loss = train_epoch(model, loaders['train'], optimizer, ema,
                                 MAX_ATOMS, DEVICE)
        scheduler.step()
        val_loss   = eval_epoch(model, loaders['valid'], MAX_ATOMS, DEVICE, ema)

        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})

        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save({
                'epoch'     : epoch,
                'model_state': model.state_dict(),
                'ema_shadow' : ema.shadow,
                'optimizer'  : optimizer.state_dict(),
            }, '/kaggle/working/edm_best.pt')

        elapsed = time.time() - t0
        print(f"{epoch:>6}  {train_loss:>11.5f}  {val_loss:>11.5f}  "
              f"{best_val:>11.5f}  {elapsed:>5.1f}s")

    # ── Save training history ─────────────────────────────────────────────────
    with open('/kaggle/working/edm_logs/history.json', 'w') as f:
        json.dump(history, f, indent=2)

    # ── Final test evaluation ─────────────────────────────────────────────────
    print(f"\n{'═' * 58}")
    print(f"Training done. Loading best checkpoint (epoch {best_epoch}) …")
    ckpt = torch.load('/kaggle/working/edm_best.pt', map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    ema.shadow = ckpt['ema_shadow']

    test_loss = eval_epoch(model, loaders['test'], MAX_ATOMS, DEVICE, ema)

    print(f"\n{'═' * 58}")
    print(f"  TEST RESULTS  —  EDM  (QM9 generation)")
    print(f"{'═' * 58}")
    print(f"  Test diffusion loss (L2, EMA) : {test_loss:.6f}")
    print(f"{'─' * 58}")
    print(f"  Best validation loss          : {best_val:.6f}  (epoch {best_epoch})")
    print(f"{'═' * 58}\n")

    summary = dict(
        test_loss      = test_loss,
        best_val_loss  = best_val,
        best_epoch     = best_epoch,
        model          = 'EDM',
        diffusion_steps= DIFFUSION_STEPS,
        noise_schedule = NOISE_SCHEDULE,
        hidden_nf      = HIDDEN_NF,
        n_layers       = N_LAYERS,
        n_params       = n_params,
    )
    with open('/kaggle/working/edm_logs/test_results.json', 'w') as f:
        json.dump(summary, f, indent=2)


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    smoke_test()
    main()
else:
    smoke_test()